# DGL vs PyG — Results Comparison

Loads the JSON files saved by `dgl_benchmark.ipynb` and `pyg_benchmark.ipynb` and shows side-by-side comparison tables.

**No ML environment required** — only `pandas` and `numpy`.

Run order:
1. `dgl_benchmark.ipynb` (DGL env)
2. `pyg_benchmark.ipynb` (PyG env)
3. This notebook (any env)

In [ ]:
import os, json
import numpy as np
import pandas as pd

OUT_DIR = os.path.join(os.getcwd(), 'comparison_outputs')

def load(filename):
    path = os.path.join(OUT_DIR, filename)
    if not os.path.exists(path):
        print(f'Missing: {path}')
        return None
    with open(path) as f:
        return json.load(f)

dgl_graph   = load('dgl_graph_stats.json')
pyg_graph   = load('pyg_graph_stats.json')
dgl_metrics = load('dgl_metrics.json')
pyg_metrics = load('pyg_metrics.json')
dgl_disease = load('dgl_disease_auroc.json')
pyg_disease = load('pyg_disease_auroc.json')

all_present = all(x is not None for x in [dgl_graph, pyg_graph, dgl_metrics, pyg_metrics, dgl_disease, pyg_disease])
print('All outputs present:', all_present)

---
## Stage 1 — Graph construction parity

In [ ]:
if dgl_graph and pyg_graph:
    # Node counts
    all_ntypes = sorted(set(dgl_graph['node_counts']) | set(pyg_graph['node_counts']))
    rows = []
    for ntype in all_ntypes:
        d = dgl_graph['node_counts'].get(ntype, '—')
        p = pyg_graph['node_counts'].get(ntype, '—')
        match = d == p
        rows.append({'node type': ntype, 'DGL': d, 'PyG': p, 'match': '✓' if match else '✗'})
    df = pd.DataFrame(rows)
    print('=== Node counts ===')
    print(df.to_string(index=False))
    print(f'  All match: {(df["match"] == "✓").all()}')

In [ ]:
if dgl_graph and pyg_graph:
    # Edge counts
    all_etypes = sorted(set(dgl_graph['edge_counts']) | set(pyg_graph['edge_counts']))
    rows = []
    for et in all_etypes:
        d = dgl_graph['edge_counts'].get(et, '—')
        p = pyg_graph['edge_counts'].get(et, '—')
        match = d == p
        rows.append({'edge type': et, 'DGL': d, 'PyG': p, 'match': '✓' if match else '✗'})
    df = pd.DataFrame(rows)
    print('=== Edge counts ===')
    print(df.to_string(index=False))
    print(f'  All match: {(df["match"] == "✓").all()}')

In [ ]:
if dgl_graph and pyg_graph:
    # Edge index parity (sorted)
    common_etypes = set(dgl_graph['edge_indices']) & set(pyg_graph['edge_indices'])
    rows = []
    for et in sorted(common_etypes):
        d = dgl_graph['edge_indices'][et]
        p = pyg_graph['edge_indices'][et]
        src_match = d['src'] == p['src']
        dst_match = d['dst'] == p['dst']
        match = src_match and dst_match
        rows.append({'edge type': et, 'match': '✓' if match else '✗',
                     'src_match': src_match, 'dst_match': dst_match})
    df = pd.DataFrame(rows)
    print('=== Edge index parity (sorted by src, dst) ===')
    print(df.to_string(index=False))
    print(f'  All match: {(df["match"] == "✓").all()}')

---
## Stage 2 — Training metric parity

In [ ]:
METRIC_TOL = 0.01   # accept ≤1 pp difference in AUROC/AUPRC

if dgl_metrics and pyg_metrics:
    print('=== Summary metrics ===')
    summary_rows = []
    for key in ['macro_auroc', 'macro_auprc', 'micro_auroc', 'micro_auprc', 'loss']:
        d = dgl_metrics[key]
        p = pyg_metrics[key]
        delta = abs(d - p)
        summary_rows.append({'metric': key, 'DGL': round(d, 4), 'PyG': round(p, 4),
                              '|Δ|': round(delta, 4),
                              'pass': '✓' if delta < METRIC_TOL else '✗'})
    print(pd.DataFrame(summary_rows).to_string(index=False))

In [ ]:
if dgl_metrics and pyg_metrics:
    print('=== Per edge-type AUROC ===')
    d_auroc = dgl_metrics['auroc_per_etype']
    p_auroc = pyg_metrics['auroc_per_etype']
    d_auprc = dgl_metrics['auprc_per_etype']
    p_auprc = pyg_metrics['auprc_per_etype']

    all_etypes = sorted(set(d_auroc) | set(p_auroc))
    rows = []
    for et in all_etypes:
        da = d_auroc.get(et, float('nan'))
        pa = p_auroc.get(et, float('nan'))
        dp = d_auprc.get(et, float('nan'))
        pp = p_auprc.get(et, float('nan'))
        delta_auc = abs(da - pa)
        delta_prc = abs(dp - pp)
        ok = delta_auc < METRIC_TOL and delta_prc < METRIC_TOL
        rows.append({
            'edge type': et,
            'DGL AUROC': round(da, 4), 'PyG AUROC': round(pa, 4), 'ΔAUROC': round(delta_auc, 4),
            'DGL AUPRC': round(dp, 4), 'PyG AUPRC': round(pp, 4), 'ΔAUPRC': round(delta_prc, 4),
            'pass': '✓' if ok else '✗',
        })
    df = pd.DataFrame(rows)
    pd.set_option('display.max_colwidth', 60)
    print(df.to_string(index=False))
    print(f'  All pass: {(df["pass"] == "✓").all()}')

---
## Stage 3 — Disease-centric AUROC parity

In [ ]:
DISEASE_TOL = 0.02

if dgl_disease and pyg_disease:
    da = dgl_disease['auroc']
    pa = pyg_disease['auroc']
    all_ids = sorted(set(da) | set(pa))
    rows = []
    for did in all_ids:
        d = da.get(did, float('nan'))
        p = pa.get(did, float('nan'))
        delta = abs(d - p)
        ok = delta < DISEASE_TOL
        rows.append({'disease id': did, 'DGL AUROC': round(d, 4), 'PyG AUROC': round(p, 4),
                     '|Δ|': round(delta, 4), 'pass': '✓' if ok else '✗'})
    df = pd.DataFrame(rows)
    print('=== Per-disease AUROC (indication) ===')
    print(df.to_string(index=False))
    print(f'  All pass: {(df["pass"] == "✓").all()}')

---
## Optional — Forward-pass score comparison

Only available if Stage 2 of `pyg_benchmark.ipynb` was run (requires DGL weights to be saved first).

In [ ]:
dgl_fwd = load('dgl_forward_scores.json') if os.path.exists(os.path.join(OUT_DIR, 'dgl_forward_scores.json')) else None
pyg_fwd = load('pyg_forward_scores.json') if os.path.exists(os.path.join(OUT_DIR, 'pyg_forward_scores.json')) else None

# Note: dgl_benchmark saves model weights but not scores by default.
# To compare forward scores, add a forward-pass cell to dgl_benchmark.ipynb
# that saves scores with the same torch.manual_seed(1).

FWD_TOL = 1e-5
if dgl_fwd and pyg_fwd:
    rows = []
    for et in sorted(set(dgl_fwd) | set(pyg_fwd)):
        d = np.array(dgl_fwd.get(et, []))
        p = np.array(pyg_fwd.get(et, []))
        if d.shape != p.shape:
            rows.append({'edge type': et, 'max |Δ|': '—', 'pass': '✗ (shape mismatch)'})
            continue
        max_diff = float(np.abs(d - p).max()) if len(d) > 0 else 0.0
        ok = max_diff < FWD_TOL
        rows.append({'edge type': et, 'max |Δ|': f'{max_diff:.2e}', 'pass': '✓' if ok else '✗'})
    print('=== Forward-pass score parity (DGL weights → PyG model) ===')
    print(pd.DataFrame(rows).to_string(index=False))
else:
    print('Forward-pass scores not found — skipped.')
    print('To enable: run Stage 2 of pyg_benchmark.ipynb after dgl_benchmark.ipynb has saved weights.')